# PlantSeg Multiclass U-Net Workflow

This notebook-style workflow converts PlantSeg binary disease masks into multiclass masks and trains U-Net to predict disease class per pixel.

Class convention:

- `0`: background
- `1..114`: compact disease classes from `Metadatav2.csv`

Use the CUDA interpreter: `F:\\PyTorch_GPU\\torch_gpu\\Scripts\\python.exe`.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path(r"F:\\PyTorch_GPU\\Plant_Seg")
SRC_DIR = PROJECT_ROOT / "plantseg_training" / "src"
sys.path.insert(0, str(SRC_DIR))

DATA_ROOT = PROJECT_ROOT / "Data_exploration" / "data" / "archive" / "plantsegv2"
CONFIG_PATH = PROJECT_ROOT / "plantseg_training" / "configs" / "unet_multiclass.json"
PROCESSED_ROOT = PROJECT_ROOT / "plantseg_training" / "processed" / "multiclass"

## 1. Dataset class research

In [ ]:
import pandas as pd

metadata = pd.read_csv(DATA_ROOT / "Metadatav2.csv")
print("images:", len(metadata))
print("plants:", metadata["Plant"].nunique())
print("disease labels:", metadata["Disease"].nunique())
print("raw index min/max:", metadata["Index"].min(), metadata["Index"].max())
display(metadata["Split"].value_counts())
display(metadata["Plant"].value_counts().head(35))
display(metadata.groupby("Plant")["Disease"].nunique().sort_values(ascending=False))
display(metadata.groupby(["Plant", "Disease"]).size().reset_index(name="images").sort_values(["Plant", "Disease"]))

## 2. Generate multiclass masks

Run this once, or rerun with `--overwrite` after changing preprocessing logic.

In [ ]:
print(r"F:\PyTorch_GPU\torch_gpu\Scripts\python.exe plantseg_training/scripts/preprocess_multiclass_masks.py --overwrite")

## 3. Inspect generated class map and weights

In [ ]:
class_map = pd.read_csv(PROCESSED_ROOT / "reports" / "class_map.csv")
pixel_counts = pd.read_csv(PROCESSED_ROOT / "reports" / "class_pixel_counts.csv")
weights = json.loads((PROCESSED_ROOT / "reports" / "class_weights.json").read_text(encoding="utf-8"))
display(class_map.head(20))
display(pixel_counts.sort_values("pixel_count", ascending=False).head(20))
display(pixel_counts.sort_values("pixel_count", ascending=True).head(20))
print("num_classes:", weights["num_classes"])
print("first weights:", weights["class_weights"][:10])

## 4. Smoke-test one multiclass sample and CUDA forward pass

In [ ]:
import torch
from plantseg_training.data import make_datasets
from plantseg_training.models import build_model

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
train_ds, val_ds, test_ds = make_datasets(config["dataset"])
sample = train_ds[0]
print(sample["pixel_values"].shape, sample["labels"].shape, sample["labels"].dtype)
print("unique labels:", torch.unique(sample["labels"])[:20])

config["model"]["loss"]["class_weights"] = weights["class_weights"]
model = build_model(config["model"]).cuda()
out = model(pixel_values=sample["pixel_values"].unsqueeze(0).cuda(), labels=sample["labels"].unsqueeze(0).cuda())
print("loss:", float(out["loss"].detach().cpu()))
print("logits:", tuple(out["logits"].shape))

## 5. Train multiclass U-Net

In [ ]:
print(r"F:\PyTorch_GPU\torch_gpu\Scripts\python.exe plantseg_training/train.py --config plantseg_training/configs/unet_multiclass.json")

## 6. Monitor

In [ ]:
print(r"nvidia-smi -l 5")
print(r"F:\PyTorch_GPU\torch_gpu\Scripts\python.exe -m mlflow ui --backend-store-uri plantseg_training/outputs/mlruns")
print(r"F:\PyTorch_GPU\torch_gpu\Scripts\python.exe -m tensorboard.main --logdir plantseg_training/outputs/unet_multiclass")